In [4]:
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query, register_integrated_taxi_trips

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('storage_metrics')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/19 13:44:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [7]:
TABLES=[
    'air_quality_vs_demand',
    'average_trip_distance_by_weather',
    'monthly_zone_demand',
    'zone_weather_demand_variation'
]

In [9]:
for i in TABLES:
    df = spark.read.format("delta").load(
        f"s3a://delta-data/analytics/{i}"
    )

    df.write \
        .mode("overwrite") \
        .option("header", "true") \
        .csv(f"s3a://delta-data/analytics/csv/{i}")

    path = spark._jvm.org.apache.hadoop.fs.Path(f"s3a://delta-data/analytics/csv/{i}")
    fs = path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration())

    csv_size = fs.getContentSummary(path).getLength()
    size = spark.sql(f"""
        DESCRIBE DETAIL delta.`s3a://delta-data/analytics/{i}`
    """).first()['sizeInBytes']
    print(i)
    print(f"size in csv: {csv_size}")
    print(f"size in delta: {size}")
    print('csv/delta')
    print(csv_size/size)
    print("---------")

26/09/19 13:48:31 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


air_quality_vs_demand
size in csv: 515092
size in delta: 38972
csv/delta
13.216976290670225
---------


26/09/19 13:48:32 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


average_trip_distance_by_weather
size in csv: 2929
size in delta: 3673
csv/delta
0.7974407841001906
---------


26/09/19 13:48:33 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


monthly_zone_demand
size in csv: 149221
size in delta: 11441
csv/delta
13.042653614194563
---------


26/09/19 13:48:34 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


zone_weather_demand_variation
size in csv: 40658
size in delta: 7736
csv/delta
5.255687693898656
---------
